# Optional Lab 7A: Trace a Real Model with Litefuse

**Module 7: Trace Analysis and Litefuse** · [Lesson](course.md) · [Lab 7](lab07_data_analysis_litefuse.ipynb) · [Quiz](quiz07_analysis_and_litefuse.ipynb)

In section 5 of the Lab, the assistant's model was simulated: it returned written answers and reported its own token counts and costs. This optional Lab gives the assistant a real model and traces it the way applications do in production: the OpenAI SDK, wrapped by Langfuse's integration, records every model call by itself, with the prompt, the answer, the model, its parameters, the tokens that the model counted, and how long the call took. Then you compare the real calls with the simulated ones in Doris and in Litefuse.

By default the model runs on your computer: the server of llama.cpp, an open-source inference engine, with Qwen2.5 0.5B Instruct, a small open model (Apache 2.0 license). It runs on the CPU and needs no account or key. Section 3 shows how to use OpenAI's API instead, with your own key.

Plan about 15 minutes, plus the first download of the server's image (about 300 MB) and of the model (491 MB). The model server uses about 0.4 GB of memory. You need Litefuse from section 4 of the Lab, and the traces of its section 5 to compare with. A real model's answers, token counts, and times vary, so yours can differ from the numbers quoted here.

### Initialize the Lab

Run this cell once before using `lab.sql(...)` or the other helpers.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Check that Litefuse runs

The next cell starts Litefuse if it does not run, as section 4 of the Lab does; when it runs, the cell only checks it.

In [ ]:
lab.start_litefuse()

**Expected result**

The card "Litefuse is running" with the address and the user to sign in with. When Litefuse already runs, the port check shows 23100 as `used by this course`, and the cell takes a few seconds.

## 2. Start the local model

`lab.start_local_model()` downloads the model file once into a Docker volume and checks its SHA-256 checksum. Then it starts llama.cpp's server, the container `doris-obs-llm` from `environments/llm/compose.llm.yml`, on 127.0.0.1:28080 (`DORIS_OBS_LLM_PORT`). The server answers OpenAI's chat completions API under `/v1`, so any OpenAI client can call it.

The model file comes from Hugging Face. If huggingface.co cannot be reached from your network, set `DORIS_OBS_LLM_MODEL_URL` to the address of the same file on a mirror before you start Jupyter; the checksum shows that the file is the same. If you use OpenAI instead (section 3), skip this cell.

In [ ]:
lab.start_local_model()

**Expected result**

- The port check: 28080 (`DORIS_OBS_LLM_PORT`) is `available`, or `used by this course` when the server already runs.
- On the first run, after the download of the server's image, a line every 5 seconds such as `Downloaded 240 MB of 491 MB`, then `/models/qwen2.5-0.5b-instruct-q4_k_m.gguf: OK`, the checksum. The download takes a minute or more, depending on your connection. Later runs print `Already downloaded` and check the file again.
- The server starts, and the table lists one model, `qwen2.5-0.5b-instruct`.
- The card "The local model is running", a few seconds after the server starts: it loads the model into memory first.

## 3. Choose the model

The assistant calls its model through the OpenAI SDK. `MODEL_PROVIDER` decides where the calls go:

- `"local"`, the default: the model of section 2. llama.cpp's server does not check API keys, but the SDK needs one, so any text does.
- `"openai"`: OpenAI's API, with the model `gpt-4o-mini`. Set your API key in the variable `OPENAI_API_KEY` before you start Jupyter, for example `OPENAI_API_KEY=sk-... .venv/bin/jupyter lab`. Do not write the key in the notebook: notebooks get saved, shared, and committed. The three questions of section 4 cost far less than one cent.

In [ ]:
MODEL_PROVIDER = "local"  # or "openai", with your own key in OPENAI_API_KEY

if MODEL_PROVIDER == "local":
    MODEL_BASE_URL, MODEL_API_KEY, MODEL_NAME = f"{lab.local_model_url}/v1", "not-checked", "qwen2.5-0.5b-instruct"
else:
    # lab.openai_api_key() reads OPENAI_API_KEY, and stops with a card when it is not set.
    MODEL_BASE_URL, MODEL_API_KEY, MODEL_NAME = "https://api.openai.com/v1", lab.openai_api_key(), "gpt-4o-mini"
print(f"The assistant calls {MODEL_NAME} at {MODEL_BASE_URL}")

**Expected result**

`The assistant calls qwen2.5-0.5b-instruct at http://127.0.0.1:28080/v1`, or `gpt-4o-mini at https://api.openai.com/v1` with `"openai"`. With `"openai"` and no key in the environment, the card "OPENAI_API_KEY is not set" says how to set it; the key itself is never shown.

## 4. Trace the assistant with the real model

The assistant is the one of section 5 of the Lab, with two differences:

- `call_model` is gone. `answer_question` calls the model with the OpenAI SDK's `client.chat.completions.create`, and asks for a one-sentence answer from the documents, with `temperature=0`, so that the same question gets the same answer.
- `from langfuse.openai import OpenAI` imports the OpenAI SDK's client wrapped by Langfuse's integration. Each call becomes a generation observation by itself, named `llm-answer` with `name=`, which the wrapper removes from the request: it records the messages, the answer, the model, the parameters, the token counts from the model's response, and the duration. Nothing is reported by hand.

The traces are named `course-assistant-real-model`, so that you can tell them from the Lab's `course-assistant` traces. The users, sessions, and questions are the same.

In [ ]:
import os
import time

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-doris-course"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-doris-course"
os.environ["LANGFUSE_BASE_URL"] = lab.litefuse_url
os.environ["OTEL_SERVICE_NAME"] = "course-assistant"

from langfuse import get_client, observe, propagate_attributes
from langfuse.openai import OpenAI  # the OpenAI SDK's client, wrapped by Langfuse to trace every call

langfuse = get_client()
client = OpenAI(base_url=MODEL_BASE_URL, api_key=MODEL_API_KEY)

# The documents that the assistant searches, by topic, as in the Lab.
DOCUMENTS = {
    "doris": "Apache Doris is a real-time analytical database. It answers SQL queries over large tables in seconds.",
    "litefuse": "Litefuse is an LLM engineering platform. It keeps its traces and observations in Apache Doris tables.",
    "collector": "The OpenTelemetry Collector receives logs, metrics, and traces and exports them to Doris.",
}


@observe(as_type="retriever", name="search-docs")
def search_docs(question):
    time.sleep(0.05)  # a lookup in a document index takes a moment
    return [text for topic, text in DOCUMENTS.items() if topic in question.lower()]


@observe(name="answer-question")
def answer_question(question, *, user_id, session_id):
    with propagate_attributes(trace_name="course-assistant-real-model", user_id=user_id, session_id=session_id):
        trace_ids.append(langfuse.get_current_trace_id())
        documents = search_docs(question)
        response = client.chat.completions.create(
            name="llm-answer",  # the observation's name in Litefuse; the wrapper does not send it to the model
            model=MODEL_NAME,
            messages=[
                {"role": "system",
                 "content": "Answer the question in one sentence, with these documents: " + " ".join(documents)},
                {"role": "user", "content": question},
            ],
            temperature=0,
        )
        return response.choices[0].message.content


trace_ids = []
for user_id, session_id, question in [
    ("user-1", "session-1", "What is Apache Doris?"),
    ("user-1", "session-1", "Where does Litefuse keep its traces?"),
    ("user-2", "session-2", "What is Apache Doris?"),
]:
    print(f"{user_id}: {question}\n  -> {answer_question(question, user_id=user_id, session_id=session_id)}")
langfuse.flush()
print("Trace IDs:", trace_ids)

**Expected result**

Three questions and the answers that the model wrote, one sentence each, then `Trace IDs:` and three IDs. The answers stay close to the documents, because the prompt asks for that: for example `Apache Doris is a real-time analytical database that answers SQL queries over large tables in seconds.` With `temperature=0`, both answers to "What is Apache Doris?" are the same. The cell takes several seconds: on a laptop's CPU, with the rest of the course's environment running, each call takes one to two seconds.

## 5. Compare the real and the simulated model calls

The next cell waits for the observations of the three traces in Doris, as in the Lab.

In [ ]:
TRACE_ID_LIST = ", ".join(f"'{trace_id}'" for trace_id in trace_ids)
real_spans = lab.wait_for_rows(
    f"""
    SELECT trace_id, name, type, provided_model_name AS model, usage_details, total_cost,
           TIMESTAMPDIFF(MICROSECOND, start_time, end_time) / 1000 AS duration_ms
    FROM litefuse.spans_llm_app
    WHERE trace_id IN ({TRACE_ID_LIST})
    ORDER BY start_time, is_root DESC
    """,
    title="The observations of the real model's traces in spans_llm_app",
    hint="Litefuse loads a batch of traces into Doris a few seconds after it receives it. Check the worker's "
         "log with docker logs doris-obs-litefuse-worker, then rerun this cell.",
    min_rows=9,
    timeout_seconds=90,
)

**Expected result**

Nine rows, three per trace. The `llm-answer` rows have the model `qwen2.5-0.5b-instruct`, a `duration_ms` of about 1,000 to 2,000, and token counts from the model, for example `{"input":50, "output":20, "total":70, "input_cached_tokens":0}`:

- The prompt has 26 words but about 50 tokens: a tokenizer splits text into pieces that are often shorter than words, and the chat format adds tokens around each message.
- `input_cached_tokens` counts the prompt tokens that the server took from its cache instead of processing them again. llama.cpp's server keeps the prompts that it processed; the third question repeats the first, so on a fresh server its call has about 49 cached tokens and only 1 new `input` token. OpenAI's API also reports cached tokens, for long prompts that it saw recently, and charges less for them.
- `total_cost` is NULL: Litefuse computes the cost of a call from its price list of known models, which has no price for a model on your computer. With `gpt-4o-mini`, the row has the cost that Litefuse computed from OpenAI's prices.

The next cell compares the model calls of all traces in the project by model: the real ones and the simulated ones of the Lab. The prompt tokens are `total` minus `output`, which includes cached tokens for both kinds of call.

In [ ]:
lab.sql("""
SELECT provided_model_name AS model,
       COUNT(*) AS calls,
       ROUND(AVG(usage_details['total'] - usage_details['output'])) AS avg_prompt_tokens,
       ROUND(AVG(usage_details['output'])) AS avg_output_tokens,
       SUM(IFNULL(usage_details['input_cached_tokens'], 0)) AS cached_tokens,
       ROUND(AVG(TIMESTAMPDIFF(MICROSECOND, start_time, end_time)) / 1000) AS avg_duration_ms,
       SUM(total_cost) AS total_cost_usd
FROM litefuse.spans_llm_app
WHERE type = 'GENERATION'
GROUP BY provided_model_name
ORDER BY model
""", title="Model calls by model");

**Expected result**

Two rows, three calls each when you ran section 5 of the Lab and section 4 of this Lab once:

| `model` | `avg_prompt_tokens` | `avg_output_tokens` | `cached_tokens` | `avg_duration_ms` | `total_cost_usd` |
| --- | --- | --- | --- | --- | --- |
| `qwen2.5-0.5b-instruct` | about 51 | about 18 | about 63 | about 1,000 to 2,000 | NULL |
| `simulated-llm` | 27 | 16 | 0 | about 165 | `0.002270000000` |

The real model counts about twice as many prompt tokens as the simulated one counted words, takes longer, and costs nothing that Litefuse knows of. Real calls also vary from run to run: rerun section 4, and the duration and the cached tokens change, while the simulated calls of the Lab always report the same numbers.

## 6. The real model calls in Litefuse

In Litefuse's web UI ([http://127.0.0.1:23100](http://127.0.0.1:23100), signed in as in section 7 of the Lab):

1. Choose **Tracing**. The list now has the three `course-assistant-real-model` traces next to the Lab's `course-assistant` traces.
2. Choose a `course-assistant-real-model` trace, then `llm-answer` in its tree. The panel shows the prompt and the answer as messages, the model `qwen2.5-0.5b-instruct`, the parameters that the wrapper recorded, such as `temperature: 0`, the latency, and the tokens. There is no cost.
3. Choose **Home**. The model charts list both models, and the latency percentiles of the generations show how much longer the real calls take.

## 7. Try it

Ask a question that the documents do not answer, such as "Who created Apache Doris?". Add it to the list of questions in section 4, rerun that cell, and open its trace in Litefuse. `search-docs` returns the document about Doris, which does not say who created it, so the model answers from what it learned in training, and a model this small is often wrong. The trace shows why: the prompt that the model received and the answer that it gave, side by side. This is what LLM traces are for: finding out, request by request, where an answer came from.

## Completion criteria

- `spans_llm_app` has the observations of the real model's traces, with the model, the token counts, and the duration that Langfuse's OpenAI integration recorded by itself.
- You can explain why the real calls count more tokens than words, what `input_cached_tokens` counts, and why the local model's calls have no cost.

## Clean up

`lab.stop_local_model()` stops the model server, which frees its memory. The model file stays in the volume `doris-obs-llm_models`, so the next `lab.start_local_model()` does not download it again. `lab.teardown()` removes the server and the volume with the rest of the environment.

## Next

Return to [Quiz 7: Trace Analysis and Litefuse](quiz07_analysis_and_litefuse.ipynb), or continue to [Module 8: Alerting with Grafana](../module08-alerting/lab08_alerting.ipynb).